# Full Pipeline Run

**AI-Based Early Detection and Classification of Foot and Nail Conditions Using Transfer Learning for Rural Healthcare**

Runs the project end to end: fetch data → extract montages → preprocess → train both models.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

Every step is guarded, so re-running a cell that has already completed skips its
work instead of redoing it. If the runtime is recycled part-way through, re-run
from the top — completed steps are restored from Drive rather than recomputed.

**Storage.** All work happens on `/content`, which is fast local disk. Finished
artefacts are packed into single archives and copied to Drive, because Drive
buffers writes and loses tens of thousands of small files when the runtime dies.
Run the `save` cells when you reach them — they are what makes the work survive.

## 1. Setup

In [12]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

REPO    = 'https://github.com/gurubasavarajharlapur-jpg/Dissertation_AI_FOOT_NAIL_DISEASE.git'
BRANCH  = 'claude/foot-nail-disease-ai-fyrdsb'
PROJECT = Path('/content/Dissertation_AI_FOOT_NAIL_DISEASE')
DRIVE   = Path('/content/drive/MyDrive/dissertation_foot_nail')

# Clone if absent, pull if present, so this cell is safe to re-run.
if (PROJECT / '.git').is_dir():
    !cd {PROJECT} && git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git pull -q --ff-only
else:
    !git clone -q --branch {BRANCH} {REPO} {PROJECT}

%cd {PROJECT}
!pip install -q -r requirements.txt
!git log --oneline -1

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/Dissertation_AI_FOOT_NAIL_DISEASE
4ae6d9d (HEAD -> claude/foot-nail-disease-ai-fyrdsb, origin/claude/foot-nail-disease-ai-fyrdsb) Identify montage sheets by their naming convention, not by location


In [13]:
# An earlier setup symlinked data/ into Drive, which is what lost the data.
# This copies anything still on Drive back to local disk, then removes the links.
!python src/colab_sync.py unlink
!python src/colab_sync.py status

replacing Drive symlinks with real local directories:
  data/raw         not a symlink, left alone
  data/processed   not a symlink, left alone
  models           not a symlink, left alone
  results          not a symlink, left alone
project : /content/Dissertation_AI_FOOT_NAIL_DISEASE
drive   : /content/drive/MyDrive/dissertation_foot_nail  (mounted)

local (/content — lost on runtime restart):
  processed        0 file(s)       0.0 B
  models           0 file(s)       0.0 B
  results          0 file(s)       0.0 B
  raw           6819 file(s)    451.7 MB

drive archives (survive restarts):
  none yet — run `save` after preprocessing or training


In [14]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow', tf.__version__, '| Keras', tf.keras.__version__)
print('GPU:', [g.name for g in gpus] if gpus else
      'NONE — set Runtime > Change runtime type > T4 GPU before the training cells')

TensorFlow 2.20.0 | Keras 3.13.2
GPU: ['/physical_device:GPU:0']


## 2. Restore anything already saved

If a previous session got as far as preprocessing or training, this brings it
back and you can skip straight to whichever step is still outstanding.

In [15]:
!python src/colab_sync.py restore

restore : processed, models, results
  processed  no archive on Drive (processed_dataset.tar)
  models     no archive on Drive (models.tar)
  results    no archive on Drive (results.tar)


## 3. What raw data is present?

In [16]:
IMAGE_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
RAW = PROJECT / 'data' / 'raw'

def count_images(path) -> int:
    path = Path(path)
    if not path.exists():
        return 0
    return sum(1 for f in path.rglob('*')
               if f.is_file() and f.suffix.lower() in IMAGE_EXT)

# Expected counts, from the runs that completed successfully.
EXPECTED = {
    'figshare_nail':       1635,
    'mendeley_foot':       5443,
    'ulcer_fuseg':         1370,
    'figshare_nail_tiles': 18093,
}

print(f"{'folder':<24}{'present':>9}{'expected':>10}")
have = {}
for name, expected in EXPECTED.items():
    n = count_images(RAW / name)
    have[name] = n
    flag = 'ok' if n >= expected * 0.95 else 'MISSING/PARTIAL'
    print(f'{name:<24}{n:>9}{expected:>10}   {flag}')

folder                    present  expected
figshare_nail                   0      1635   MISSING/PARTIAL
mendeley_foot                5443      5443   ok
ulcer_fuseg                  1370      1370   ok
figshare_nail_tiles             0     18093   MISSING/PARTIAL


## 4. Fetch raw data

Each cell skips itself if the data is already there.

In [17]:
# Figshare onychomycosis: ~2.2 GB
if have['figshare_nail'] < EXPECTED['figshare_nail'] * 0.95:
    !python src/download_data.py --source figshare
    have['figshare_nail'] = count_images(RAW / 'figshare_nail')
else:
    print('figshare_nail already present — skipping')
print('figshare_nail:', have['figshare_nail'])


=== Figshare 5398573 (onychomycosis / nail) ===
3 file(s), 2.2 GB total
  - dataset (A1) - thumbnail (1.7GB).zip                        1.7 GB
  - dataset (A2) - thumbnail.zip                              105.0 MB
  - datasets (B1, B2, C, D, E).zip                            358.2 MB
                                                           
[FAILED] Figshare 5398573 (onychomycosis / nail)
'dataset (A1) - thumbnail (1.7GB).zip' downloaded as 0 bytes from https://ndownloader.figshare.com/files/9302500
  The server accepted the request but sent no content. Download the
  dataset manually (see --help) and unzip it into data/raw/.

1 of 1 source(s) failed. Use the manual steps in `--help` for those, then run `python src/inspect_data.py`.
figshare_nail: 0


In [18]:
# FUSeg / AZH foot ulcers: ~600 MB, images only (the labels/ folders are
# segmentation masks and must never enter a classification training set).
import shutil

if have['ulcer_fuseg'] < EXPECTED['ulcer_fuseg'] * 0.95:
    !rm -rf /tmp/ulcer_repo
    !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/uwm-bigdata/wound-segmentation.git /tmp/ulcer_repo
    !cd /tmp/ulcer_repo && git sparse-checkout set --no-cone '/data/**/images/**'

    SRC  = Path('/tmp/ulcer_repo/data')
    DEST = RAW / 'ulcer_fuseg'
    copied = 0
    for img_dir in sorted(SRC.rglob('images')):
        parts = img_dir.relative_to(SRC).parts
        tag = ('fuseg' if 'Foot Ulcer' in parts[0] else 'medetec') + '_' + parts[-2]
        out = DEST / tag
        out.mkdir(parents=True, exist_ok=True)
        for f in img_dir.iterdir():
            if f.suffix.lower() in IMAGE_EXT:
                shutil.copy2(f, out / f.name)
                copied += 1
        print(f'  {tag:<18} {len(list(out.iterdir())):>5}')
    shutil.rmtree('/tmp/ulcer_repo', ignore_errors=True)
    have['ulcer_fuseg'] = count_images(DEST)
else:
    print('ulcer_fuseg already present — skipping')
print('ulcer_fuseg:', have['ulcer_fuseg'])

ulcer_fuseg already present — skipping
ulcer_fuseg: 1370


In [19]:
# Mendeley foot images, from the zips uploaded to Drive.
import zipfile

MASKS = PROJECT / 'data' / 'mendeley_masks'

# wound_mask.zip holds SEGMENTATION MASKS, not photographs. They go outside
# data/raw so neither the inspector nor training ever sees them — a binary mask
# counted as a training image is a labelled example of nothing. They are kept
# rather than deleted, being useful for cropping to the wound later.
TARGETS = {
    'Normal.zip':     RAW / 'mendeley_foot' / 'Normal',
    'wound_main.zip': RAW / 'mendeley_foot' / 'wound_main',
    'wound_mask.zip': MASKS / 'wound_mask',
}

def safe_extract(archive: Path, target: Path) -> int:
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as zf:
        members = []
        for name in zf.namelist():
            p = Path(name)
            if p.is_absolute() or '..' in p.parts:
                raise RuntimeError(f'unsafe entry in {archive.name}: {name}')
            if name.startswith('__MACOSX/') or Path(name).name.startswith('._'):
                continue
            members.append(name)
        zf.extractall(target, members=members)
    return count_images(target)

if have['mendeley_foot'] < EXPECTED['mendeley_foot'] * 0.95:
    missing = [z for z in TARGETS if not (DRIVE / z).exists()]
    if missing:
        print('Missing from Drive:', missing)
        print(f'Upload them to {DRIVE}/ and re-run this cell.')
        !ls -la {DRIVE}/*.zip
    else:
        for zip_name, target in TARGETS.items():
            n = safe_extract(DRIVE / zip_name, target)
            role = 'MASKS — excluded from training' if 'mask' in zip_name else 'training data'
            print(f'  {zip_name:<16} {n:>5} images   [{role}]')
        have['mendeley_foot'] = count_images(RAW / 'mendeley_foot')
else:
    print('mendeley_foot already present — skipping')
print('mendeley_foot:', have['mendeley_foot'])

mendeley_foot already present — skipping
mendeley_foot: 5443


In [21]:
%cd /content/Dissertation_AI_FOOT_NAIL_DISEASE
!git pull --ff-only

/content/Dissertation_AI_FOOT_NAIL_DISEASE
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.66 KiB | 847.00 KiB/s, done.
From https://github.com/gurubasavarajharlapur-jpg/Dissertation_AI_FOOT_NAIL_DISEASE
   4ae6d9d..99811ef  claude/foot-nail-disease-ai-fyrdsb -> origin/claude/foot-nail-disease-ai-fyrdsb
Updating 4ae6d9d..99811ef
Fast-forward
 src/download_data.py | 23 +++++++++++++++++------
 1 file changed, 17 insertions(+), 6 deletions(-)


In [22]:
# What does that URL actually return? Three clients, one answer.
URL = 'https://ndownloader.figshare.com/files/9302500'

print('--- curl (headers only) ---')
!curl -sIL -o /dev/null -w "status=%{http_code} size=%{size_download} type=%{content_type} redirects=%{num_redirects} final=%{url_effective}\n" "{URL}"

print('\n--- python requests, default UA ---')
import requests
r = requests.get(URL, stream=True, timeout=60)
print('status', r.status_code, '| length', r.headers.get('content-length'),
      '| type', r.headers.get('content-type'), '| redirects', len(r.history))
print('first bytes:', next(r.iter_content(64), b'')[:64])
r.close()

--- curl (headers only) ---
status=000 size=0 type= redirects=0 final=http://URL/

--- python requests, default UA ---
status 200 | length 1827905879 | type binary/octet-stream | redirects 1
first bytes: b'PK\x03\x04\x14\x00\x00\x00\x00\x00k\xb7,K\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00!\x00\x00\x00dataset (A1) - thumbnail (1.7GB)/P'


In [23]:
from pathlib import Path
DL = Path('/content/Dissertation_AI_FOOT_NAIL_DISEASE/data/raw/_downloads')
DL.mkdir(parents=True, exist_ok=True)

FILES = {
    9302500: 'dataset_A1_thumbnail.zip',
    9302503: 'dataset_A2_thumbnail.zip',
    9302506: 'datasets_B1_B2_C_D_E.zip',
}
for file_id, name in FILES.items():
    !wget -c -q --show-progress -O "{DL}/{name}" "https://ndownloader.figshare.com/files/{file_id}"

!ls -lh {DL}

/content/Dissertati 100%[===================>]   1.70G   112MB/s    in 22s     
/content/Dissertati 100%[===================>] 105.00M  95.3MB/s    in 1.1s    
/content/Dissertati 100%[===================>] 358.24M  41.0MB/s    in 5.7s    
total 2.2G
-rw-r--r-- 1 root root    0 Sep 12 18:58 'dataset (A1) - thumbnail (1.7GB).zip'
-rw-r--r-- 1 root root 1.8G Apr 10  2021  dataset_A1_thumbnail.zip
-rw-r--r-- 1 root root 106M Apr 10  2021  dataset_A2_thumbnail.zip
-rw-r--r-- 1 root root    0 Sep 12 18:58 'dataset (A2) - thumbnail.zip'
-rw-r--r-- 1 root root 359M Apr 10  2021  datasets_B1_B2_C_D_E.zip
-rw-r--r-- 1 root root    0 Sep 12 18:58 'datasets (B1, B2, C, D, E).zip'


In [24]:
from pathlib import Path

DL = Path('/content/Dissertation_AI_FOOT_NAIL_DISEASE/data/raw/_downloads')
for z in DL.glob('*.zip'):
    if z.stat().st_size == 0:
        print('removing empty:', z.name)
        z.unlink()

print()
for z in sorted(DL.glob('*.zip')):
    print(f'{z.stat().st_size/1e6:>8.0f} MB  {z.name}')

removing empty: dataset (A2) - thumbnail.zip
removing empty: datasets (B1, B2, C, D, E).zip
removing empty: dataset (A1) - thumbnail (1.7GB).zip

    1828 MB  dataset_A1_thumbnail.zip
     110 MB  dataset_A2_thumbnail.zip
     376 MB  datasets_B1_B2_C_D_E.zip


In [25]:
import zipfile
from pathlib import Path

DL  = Path('/content/Dissertation_AI_FOOT_NAIL_DISEASE/data/raw/_downloads')
RAW = Path('/content/Dissertation_AI_FOOT_NAIL_DISEASE/data/raw/figshare_nail')
RAW.mkdir(parents=True, exist_ok=True)

for z in sorted(DL.glob('*.zip')):
    if not zipfile.is_zipfile(z):
        print(f'SKIP {z.name} — not a readable zip')
        continue
    print(f'extracting {z.name} ({z.stat().st_size/1e6:.0f} MB)...')
    with zipfile.ZipFile(z) as zf:
        members = [n for n in zf.namelist()
                   if not n.startswith('__MACOSX/') and '..' not in Path(n).parts]
        zf.extractall(RAW, members=members)

n = sum(1 for p in RAW.rglob('*')
        if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
print(f'\nfigshare_nail: {n} images (expected ~1635)')

extracting dataset_A1_thumbnail.zip (1828 MB)...
extracting dataset_A2_thumbnail.zip (110 MB)...
extracting datasets_B1_B2_C_D_E.zip (376 MB)...

figshare_nail: 1635 images (expected ~1635)


In [26]:
%cd /content/Dissertation_AI_FOOT_NAIL_DISEASE
!python src/extract_montages.py
!python src/preprocessing.py --dry-run

/content/Dissertation_AI_FOOT_NAIL_DISEASE
1635 file(s) under /content/Dissertation_AI_FOOT_NAIL_DISEASE/data/raw/figshare_nail
277 follow the montage naming convention ('...#<label>.png'); 1358 are per-image files
sheet labels found:
  -focus                  31  -> None
  -etc                    29  -> UNMAPPED
  naildystrophy           28  -> None
  melanonychia            28  -> UNMAPPED
  onychomycosis           27  -> UNMAPPED
  -atypical               21  -> UNMAPPED
  normalnail              21  -> healthy
  whitespot               21  -> UNMAPPED
  nodule                  21  -> UNMAPPED
  onycholysis             20  -> UNMAPPED
  -atypical2               3  -> UNMAPPED
  pincer                   2  -> UNMAPPED
  fluco                    2  -> UNMAPPED
  tinea                    2  -> UNMAPPED
  scio_onychomycosis       2  -> UNMAPPED
  2006 undetermined        1  -> UNMAPPED
  2007 undetermined        1  -> UNMAPPED
  2008 undetermined        1  -> UNMAPPED
  2009 undetermine

## 6. Preprocess

**Save now.** This is ~250 MB as a single archive, and it is what you would
otherwise have to rebuild after a runtime restart.

In [27]:
!python src/preprocessing.py
!python src/colab_sync.py save --what processed

seed=42  target size=(224, 224)  classes=['healthy', 'nail_fungal', 'foot_wound', 'foot_ulcer']

[scan data/raw         ] kept  26544
[map to classes        ] kept  25689   dropped   855   6 folder(s) excluded or unmapped
      excluded: figshare_nail/datasets (B1, B2, C, D, E)/D/naildystrophy    439
      excluded: figshare_nail/dataset (A1) - thumbnail (1.7GB)             259
      excluded: figshare_nail/datasets (B1, B2, C, D, E)/C/naildystrophy     68
      excluded: figshare_nail/datasets (B1, B2, C, D, E)/B1/naildystrophy     50
      excluded: figshare_nail/datasets (B1, B2, C, D, E)/B2/naildystrophy     21
      excluded: figshare_nail/dataset (A2) - thumbnail                      18

   per-class after mapping:
      healthy         20853
      foot_wound       2686
      foot_ulcer       1370
      nail_fungal       780

  verifying: 100% 25689/25689 [00:04<00:00, 5150.90img/s]
[clean: readable       ] kept  25689   unreadable or truncated
[clean: min size       ] kept  2568

## 8. Train ResNet50 (comparison model)

In [ ]:
!python src/train.py --model mobilenetv2
!python src/colab_sync.py save --what models results
!python src/train.py --model resnet50
!python src/colab_sync.py save --what models results

TensorFlow 2.20.0 | Keras 3.13.2 | GPU: ['/physical_device:GPU:0']

TRAINING mobilenetv2
2026-09-12 19:28:47.930911: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1789241327.932383   10115 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
train=5862  val=1257  batch=32
class weights: healthy=0.56  nail_fungal=2.69  foot_wound=0.83  foot_ulcer=1.57
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step

backbone layers: 154   total parameters: 2,263,108

--- Stage 1: head training (15 epochs @ lr=0.001) ---
Epoch 1/15
2026-09-12 19:29:14.314223: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution,

## 9. Where things stand

In [ ]:
!python src/colab_sync.py status
print()
!ls -la models/ results/figures/

---

Send the output of the preprocessing and training cells back to Claude Code, and
Phase 5 (evaluation: confusion matrices, the MobileNetV2 vs ResNet50 comparison,
efficiency metrics and Grad-CAM) can be built against the real results.